

  
  
  
Save as SVGSave as PNGView SourceView Compiled VegaOpen in Vega Editor

In [1]:
(async function(){
  
  var vegaModule = await import("https://cdn.jsdelivr.net/npm/vega@5");
  var vegaLiteModule = await import("https://cdn.jsdelivr.net/npm/vega-lite@5");
  var vegaEmbedModule = await import("https://cdn.jsdelivr.net/npm/vega-embed@6");

  console.dir(vegaEmbed.default)
  
  /*
    Constants
  */
  var Constants = {
    speechBonus: 1.1,
    listenBonus: 1.1,
  }
  
  function choice(l=[]){
    return l[Math.floor(Math.random() * l.length)];
  }
  
  function mean(l=[]){
    var sum = 0.0;
    for(var i=0; i<l.length; i++){
      sum += l[i];
    }
    return sum/l.length;
  }
  
  /*
    Golem
  */
  function Golem(){
    this.health = Math.random();
    this.lapses = 0.0;
  }
  
  Golem.prototype.speak = function(){
    var impact = (choice([-1, 1]) * Math.random() * Constants.speechBonus);
    this.health += impact;
    return impact;
  }
  
  Golem.prototype.age = function(){
    this.lapses += 1.0;
    return this.lapses >= 1000.0;
  }
  
  Golem.prototype.listenTo = function(speech){
    var impact = (Math.random() * speech) + Constants.listenBonus;
    this.health *= impact;
  }
  
  /*
    Settings
  */
  function Setting(n){
    this.golems = [];
    for(var i=0; i<n; i++){
      this.golems.push(new Golem());
    }
  }
  
  Setting.prototype.historize = function(fact){
    if(this.history === undefined){
      this.history = {};
    }
    
    for(var attribute of Object.keys(fact)){
      if(this.history[attribute] === undefined){
        this.history[attribute] = [];
      }
      
      this.history[attribute].push(fact[attribute]);
    }
  }
  
  Setting.prototype.round = function(){
    var speaker = choice(this.golems);
    var listener = choice(this.golems);
  
    var speech = speaker.speak();
    listener.listenTo(speech);
  
    if(listener.health > 0.075){
      this.golems.push(new Golem());
    }
    
    // each golem ages
    this.golems.forEach((golem, i) => {
      if(golem.age()){
        this.golems.splice(i, 1);
      }
    })
    var meanHealth = mean(this.golems.map(g => g.health));
    var meanAge = mean(this.golems.map(g => g.lapses));
    
    this.historize({
      'round'      : this.history.round.length, 
      'mean health': meanHealth,
      'count'      : this.golems.length,
      'mean age'   : meanAge
    });
    
    return mean(this.golems.map(g => g.health));
  }
  
  var setting = new Setting(10)
  
  var chart = document.querySelector('.cell:first-child > .output')
  
  vegaEmbed(".cell:first-child>.output", {
    $schema: 'https://vega.github.io/schema/vega-lite/v5.json',
    data: {name: 'table'},
    width: 580,
    mark: 'point',
    encoding: {
      x: {field: 'round', type: 'ordinal', scale: {zero: false}},
      y: {field: 'mean health', type: 'quantitative'}
    }
  }).then(function(res) {
    function newGenerator(){
      var counter = -1;
      var previousY = [0];
    
      return function(){
        counter++;
        
        var newVals = previousY.map(function(v, c){
          return {
            'round': counter,
            'mean health': setting.round()
          }
        });
        
        previousY = newVals.map(function(v){
          return v['mean health'];
        });
        
        return newVals;
      };
    }
  
    var valueGenerator = newGenerator();
    var minimumX = -20;
    window.setInterval(function () {
      minimumX++;
      var changeSet = vega
        .changeset()
        .insert(valueGenerator())
        .remove(function (t) {
          return t['round'] < minimumX;
        });
      res.view.change('table', changeSet).run();
    }, 100);
  });
  
  setting.historize({round: 0});
})();


WARNx-scale's "zero" is dropped as it does not work with point scale.WARNInfinite extent for field "mean health": [Infinity, -Infinity]

{}